# SEN122A Statistical Analysis of Choice Behaviour

## `Lab session 01C:`
## `Modelling choice behaviour in the context of labelled data and unavailable alternatives`

**Delft University of Technology**<br>
**Q2 2026**<br>
**Instructor:** Sander van Cranenburgh<br>
**TA:** Gabriel Nova

### `Instructions`

This is an independent, exam-style lab. You are given a dataset, its context, and a set of tasks. Decide for yourself how to structure the analysis and which code from Labs 01A and 01B you can reuse. The notebook deliberately contains less guidance than the earlier labs. The lab is not graded and does not have to be submitted.

In this lab session, **your task** is to estimate and interpret choice models with alternative-specific constants (ASCs) and availability conditions. You will need to use your knowledge of choice model analysis, as well as your programming skills in Python, to complete the tasks.

**`Learning objectives lab session 01C`**

After completing the following lab session, you will be able to:
* Independently conduct a choice model analysis in Python, including:
  * Data preparation and cleaning
  * Model specification and estimation
  * Model evaluation and interpretation of results
* Use ASCs and availability conditions in choice model analysis effectively and appropriately.

**`This lab consists of 4 parts:`**

**Part 1**: Load and explore the data set<br>
**Part 2**: Estimate linear-additive RUM-MNL models<br>
**Part 3**: Test and interpret the models with ASCs<br>
**Part 4**: Test and interpret the models with availability conditions<br>

### `Use of AI tools`
AI tools, such as ChatGPT and GitHub Copilot, are powerful tools to assist with programming. Moreover, in your future careers, you will work in a world where such tools are widely available. As such, we **encourage** you to learn to use AI tools **effectively**.

However, remember that the **primary objective of these lab sessions is not to complete the assignments, but to learn and develop your own understanding and skills**. AI tools can help you achieve this, but they can also undermine your learning if you rely on them too heavily, see this [article](../../Assets/chatbots-and-thinking-english.pdf). Being able to instruct an AI tool to solve a problem is not the same as understanding how to solve it yourself. Understanding requires some effort and friction. You should therefore remain actively involved in conceptualising problems, structuring analyses, interpreting results, and critically evaluating AI-generated solutions.

We recommend using AI tools **carefully**, particularly because you will not have access to them during the `practical exam` (as internet access will be restricted). **Therefore, at all times, make sure you understand what you are doing, not just that your code works.**

### `Documentation hub`
During the **exam**, you will have **access** to the **Documentation hub**. This hub contains course materials and other useful resources, such as the lab session solutions and formula cheat sheets.

## Dataset and context

The Swissmetro stated-choice survey asked travellers to choose between **train**, **Swissmetro**, and **car** for intercity trips in Switzerland. Swissmetro was a proposed high-speed transport system. 



| Variable | Description |
|---|---|
| `GROUP` | Population group. |
| `SURVEY` | Survey location: `0` = train, `1` = car. |
| `SP` | Stated-preference survey indicator; fixed to `1`. |
| `ID` | Respondent identifier. |
| `CHOICE` | Observed choice: `0` = unknown, `1` = train, `2` = Swissmetro, `3` = car. |
| `SM_AV` | Swissmetro availability indicator. |
| `TRAIN_AV` | Train availability conditioned on valid stated-preference records. |
| `CAR_AV` | Car availability conditioned on valid stated-preference records. |
| `TRAIN_TT_SCALED` | Train travel time in **hundreds of minutes** (1 unit = 100 min). |
| `TRAIN_COST_SCALED` | Adjusted train fare in **hundreds of CHF** (1 unit = 100 CHF). |
| `TRAIN_HE` | Train headway in minutes. For example, two trains per hour correspond to a headway of 30 minutes. |
| `SM_TT_SCALED` |Swissmetro travel time in **hundreds of minutes** (1 unit = 100 min). |
| `SM_COST_SCALED` | Adjusted Swissmetro fare in **hundreds of CHF** (1 unit = 100 CHF). |
| `SM_HE` | Swissmetro headway in minutes. For example, two services per hour correspond to a headway of 30 minutes. |
| `SM_SEATS` | Swissmetro seating configuration: `0` = no airline-style seating, `1` = airline-style seating. |
| `CAR_TT_SCALED` | Car travel time in **hundreds of minutes** (1 unit = 100 min). |
| `CAR_CO_SCALED` | Car travel cost in **hundreds of CHF** (1 unit = 100 CHF). |
| `PURPOSE` | Travel purpose: `1` = commuting, `2` = shopping, `3` = business, `4` = leisure, `5` = return from work, `6` = return from shopping, `7` = return from business, `8` = return from leisure, `9` = other. |
| `FIRST` | First-class traveller: `0` = no, `1` = yes. |
| `TICKET` | Ticket type: `0` = none, `1` = two-way with half-price card, `2` = one-way with half-price card, `3` = two-way normal price, `4` = one-way normal price, `5` = half-day ticket, `6` = annual season ticket, `7` = junior or senior annual season ticket, `8` = free travel after 19:00 card, `9` = group ticket, `10` = other. |
| `WHO` | Who pays for the trip: `0` = unknown, `1` = traveller, `2` = employer, `3` = shared equally. |
| `LUGGAGE` | Luggage: `0` = none, `1` = one piece, `3` = several pieces. |
| `AGE` | Age class: `1` = age ≤ 24, `2` = 24 < age ≤ 39, `3` = 39 < age ≤ 54, `4` = 54 < age ≤ 65, `5` = age > 65. |
| `MALE` | Traveller’s gender: `0` = female, `1` = male. |
| `INCOME` | Annual income class in thousands of CHF: `0` or `1` = below 50, `2` = 50–100, `3` = above 100, `4` = unknown. |
| `GA` | Ownership of a Swiss annual season ticket covering rail and most local public transport: `0` = no, `1` = yes. |
| `ORIGIN` | Trip origin, represented by a Swiss canton code. |
| `DEST` | Trip destination, represented by a Swiss canton code. |


The file is available at `data/swissmetro.csv`.

#### Run this cell to import the necessary libraries

In [1]:
# Do not modify this cell.
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import biogeme.database as db
from biogeme.expressions import Beta, Variable


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from utils.bio_estimation_fcns import estimate_mnl, print_results

pd.set_option('display.max_columns', None)

### `Part 1: Load and explore the data`

Load the dataset. 

In [2]:
# Create that path to the data file
data_path = Path('data/swissmetro.csv')

# Load the choice data as a pandas dataframe
df = pd.read_csv(data_path, sep=',')
df.head()

,ID,CHOICE,TRAIN_AV,SM_AV,CAR_AV,TRAIN_TT_SCALED,TRAIN_COST_SCALED,TRAIN_HE,SM_TT_SCALED,SM_COST_SCALED,SM_HE,CAR_TT_SCALED,CAR_CO_SCALED,PURPOSE,FIRST,TICKET,WHO,LUGGAGE,AGE,MALE,INCOME,GA,ORIGIN,DEST
0,1,2,1,1,1,1.12,0.48,120,0.63,0.52,20,1.17,0.65,1,0,1,1,0,3,0,2,0,2,1
1,1,2,1,1,1,1.03,0.48,30,0.60,0.49,10,1.17,0.84,1,0,1,1,0,3,0,2,0,2,1
2,1,2,1,1,1,1.30,0.48,60,0.67,0.58,30,1.17,0.52,1,0,1,1,0,3,0,2,0,2,1
3,1,2,1,1,1,1.03,0.40,30,0.63,0.52,20,0.72,0.52,1,0,1,1,0,3,0,2,0,2,1
4,1,2,1,1,1,1.30,0.36,60,0.63,0.42,20,0.90,0.84,1,0,1,1,0,3,0,2,0,2,1


`1` Report the number of observations retained for estimation


In [3]:
# Print the sample counts
print(f'Observations retained for estimation:  {len(df):,}')

Observations retained for estimation:  6,759


#### `Answers`

A total of 6,759 observations are retained for model estimation. 

The dataset was pre-processed by removing non-valid choices (CHOICE = 0) and retaining only commuter and business trips (PURPOSE in [1, 3]).

`2` Report the market share of each alternative.

In [4]:
# Count the number of times each alternative is chosen
choice_counts = df['CHOICE'].value_counts(True).sort_index()
choice_counts*100

CHOICE
1    13.300784
2    60.511910
3    26.187306
Name: proportion, dtype: float64

#### `Answers`

`2` Report the market share of each alternative.

Across the 6,759 choice tasks, Swissmetro dominates the modal split with an observed market share of 60.51% (4,090 choices). Car represents the second most frequent choice at 26.19% (1,770 choices). Finally, train has the smallest market share, chosen in only 13.30% of the observations (908 choices). 


`3` For each alternative, report the number of observations in which it is unavailable. 

In [5]:
# Count the number of observations where each alternative is unavailable (AV_* == 0)
(df[['TRAIN_AV', 'SM_AV', 'CAR_AV']] == 0).sum()


TRAIN_AV       0
SM_AV          0
CAR_AV      1152
dtype: int64

#### `Answers`

Both train and Swissmetro are available across all 6,759 choice tasks. By contrast, car is unavailable in 1,152 observations, representing 17.04% of the choice situations. This unavailability captures respondents who do not possess a driving licence or do not have access to a private car for their trip. 

### `Part 2: Linear-additive RUM-MNL models`

Estimate **Model 1** with generic parameters for travel time, cost, and headway. 
<div style="text-align: left;">

$$
\begin{aligned}
V_{train} &= \beta_{time}TT_{train}+\beta_{cost}CO_{train}+\beta_{headway}HE_{train}\\
V_{SM} &= \beta_{time}TT_{SM}+\beta_{cost}CO_{SM}+\beta_{headway}HE_{SM}\\
V_{car} &= \beta_{time}TT_{car}+\beta_{cost}CO_{car}
\end{aligned}
$$
</div>

`4` Report the final log-likelihood of **Model 1** (use two decimals)

In [6]:
from biogeme.expressions import Variable

# Create the Biogeme database using the cleaned dataset
biodata = db.Database('swissmetro', df)

# Define the variables
TRAIN_TT_SCALED   = Variable('TRAIN_TT_SCALED')
TRAIN_COST_SCALED = Variable('TRAIN_COST_SCALED')
TRAIN_HE          = Variable('TRAIN_HE')
SM_TT_SCALED      = Variable('SM_TT_SCALED')
SM_COST_SCALED    = Variable('SM_COST_SCALED')
SM_HE             = Variable('SM_HE')
CAR_TT_SCALED     = Variable('CAR_TT_SCALED')
CAR_CO_SCALED     = Variable('CAR_CO_SCALED')
CHOICE            = Variable('CHOICE')

# Give a name to the model
model_name = 'Model 1: Linear-additive RUM-MNL with generic parameters'

# Parameters to be estimated
B_time    = Beta('B_time'   , 0, None, None, 0)
B_cost    = Beta('B_cost'   , 0, None, None, 0)
B_headway = Beta('B_headway', 0, None, None, 0)

# Definition of the utility functions
v_train = B_time * TRAIN_TT_SCALED + B_cost * TRAIN_COST_SCALED  + B_headway * TRAIN_HE
v_sm    = B_time * SM_TT_SCALED    + B_cost * SM_COST_SCALED     + B_headway * SM_HE
v_car   = B_time * CAR_TT_SCALED   + B_cost * CAR_CO_SCALED 

# Associate utility functions with the numbering of alternatives
V = {1: v_train, 2: v_sm, 3: v_car}

# Associate availability conditions with the numbering of alternatives
AV = {1: 1, 2: 1, 3: 1}

# Estimate the model using the estimate_mnl function
results_model1 = estimate_mnl(V, AV, CHOICE, biodata, model_name)

# Print estimation results
print_results(results_model1)

# Report final log-likelihood
LL_model1 = results_model1.data.logLike
print(f'\nFinal log-likelihood of Model 1: {LL_model1:.2f}')


Estimation finished
Cause of termination           : Relative gradient = 3.4e-09 <= 0.00012
Number of iterations           :            4
Estimation time                :      0:00:00
Number of estimated parameters :            3
Sample size                    :         6759
Excluded observations          :            0
Null log likelihood            :     -7425.52
Init log likelihood            :     -7425.52
Final log likelihood           :     -6737.02
Rho-square for the null model  :        0.093
Bayesian Information Criterion :     13500.50
Number of threads              :           11

            Value Rob. Std err Rob. t-test Rob. p-value
B_cost    -0.1594       0.0388       -4.11         0.00
B_headway -0.0051       0.0004      -11.74         0.00
B_time    -0.7896       0.0384      -20.57         0.00

Final log-likelihood of Model 1: -6737.02


#### `Answers`

The final log-likelihood of Model 1 is -6,737.02. 

`5` Interpret the estimates and their significance levels.

#### `Answers`

All three generic parameter estimates in Model 1 show negative signs consistent with theory and are statistically significant. 

- The travel time parameter indicates that an increase of 100 minutes in travel time reduces systematic utility by 0.79 units. 
- The travel cost and headway coefficients confirm that travellers percive disutility from higher fares and longer departure intervals. 

### `Part 3: Test the contribution of the ASCs`

Estimate **Model 2**, which is identical to **Model 1** except that it excludes alternative-specific constants. 

<div style="text-align: left;">

$$
\begin{aligned}
V_{train} &= ASC_{train} + \beta_{time}TT_{train}+\beta_{cost}CO_{train}+\beta_{headway}HE_{train}\\
V_{SM} &= ASC_{SM} + \beta_{time}TT_{SM}+\beta_{cost}CO_{SM}+\beta_{headway}HE_{SM}\\
V_{car} &= ASC_{car} + \beta_{time}TT_{car}+\beta_{cost}CO_{car}
\end{aligned}
$$

</div>

In [7]:
# Give a name to the model
model_name = 'Model 2: Linear-additive RUM-MNL with generic parameters and alternative-specific constants'

# Parameters to be estimated
asc_train = Beta('asc_train', 0, None, None, 0)
asc_sm    = Beta('asc_sm' ,   0, None, None, 0)
asc_car   = Beta('asc_car',   0, None, None, 1) # we fix one as reference

# Definition of the utility functions
v_train = asc_train + B_time * TRAIN_TT_SCALED + B_cost * TRAIN_COST_SCALED  + B_headway * TRAIN_HE
v_sm    = asc_sm    + B_time * SM_TT_SCALED    + B_cost * SM_COST_SCALED     + B_headway * SM_HE
v_car   = asc_car   + B_time * CAR_TT_SCALED   + B_cost * CAR_CO_SCALED 

# Associate utility functions with the numbering of alternatives
V = {1: v_train, 2: v_sm, 3: v_car}

# Associate availability conditions with the numbering of alternatives
AV = {1: 1, 2: 1, 3: 1}

# Estimate the model using the estimate_mnl function
results_model2 = estimate_mnl(V, AV, CHOICE, biodata, model_name)

# Print estimation results
print_results(results_model2)

# Report final log-likelihood
LL_model2 = results_model2.data.logLike
print(f'\nFinal log-likelihood of Model 2: {LL_model2:.2f}')


Estimation finished
Cause of termination           : Relative gradient = 4.8e-06 <= 0.00012
Number of iterations           :            4
Estimation time                :      0:00:00
Number of estimated parameters :            5
Sample size                    :         6759
Excluded observations          :            0
Null log likelihood            :     -7425.52
Init log likelihood            :     -6737.02
Final log likelihood           :     -6077.74
Rho-square for the null model  :        0.182
Bayesian Information Criterion :     12199.57
Number of threads              :           11

            Value Rob. Std err Rob. t-test Rob. p-value
B_cost    -0.6637       0.0481      -13.79         0.00
B_headway -0.0053       0.0010       -5.40         0.00
B_time    -0.1263       0.0340       -3.71         0.00
asc_sm     1.0958       0.0389       28.16         0.00
asc_train -0.1804       0.0772       -2.34         0.02

Final log-likelihood of Model 2: -6077.74


`6` Report the final log-likelihood of **Model 2** (use two decimals)

#### `Answers`


The final log-likelihood of Model 2 is -6,077.74. 

`7` Interpret the parameter estimates and their significance levels. In your interpretation, identify which attributes are already accounted for in the utility functions and explain what the ASCs capture.

#### `Answers`

- With car designated as the reference alternative ($\text{ASC}_{\text{car}} = 0$), both estimated alternative-specific constants are statistically significant.  The Swissmetro constant is strongly positive ($\text{ASC}_{\text{sm}} = 1.0958$, $t = 28.16$), whereas the train constant is negative ($\text{ASC}_{\text{train}} = -0.1804$, $t = -2.34$). 

- Travel time, travel cost, and headway parameters retain their expected negative signs and remain statistically significant.

Because travel time, cost, and headway are already controlled for in the utilities, the ASCs capture the net impact of unobserved modal attributes. These parameters thus reveal an inherent preference for Swissmetro over driving, but an intrinsic disadvantage for train, driven by unmodelled factors such as technological novelty.


`8` Explain why at least one ASC must be fixed. What problem would arise if all three ASCs were estimated?

#### `Answers`

In Random Utility Maximisation models, choice probabilities depend strictly on differences between utility levels rather than absolute utilities. Adding an arbitrary constant to all three utility equations leaves choice probabilities and the log-likelihood entirely unchanged because the constant cancels out in the logit formula. If all three alternative-specific constants were estimated freely, the model would exhibit exact multicollinearity, resulting in a singular Hessian matrix and an unidentifiable parameter vector. Fixing one constant to zero anchors the utility scale to a reference alternative and ensures identification during estimation.

`9` Compare **Model 1** and **Model 2** using a likelihood-ratio test. State the null hypothesis, calculate the test statistic, and draw a conclusion at the 5% significance level.

In [8]:
from scipy.stats import chi2

# Likelihood Ratio Test between Model 1 (restricted) and Model 2 (unrestricted)
# H0: ASC_train = 0 and ASC_sm = 0 (Model 1 is preferred)
# H1: At least one ASC is non-zero (Model 2 is preferred)

LL1 = results_model1.data.logLike
LL2 = results_model2.data.logLike

# Compute Likelihood Ratio Statistic
LRS = -2 * (LL1 - LL2)

# Degrees of freedom: difference in number of estimated parameters (5 - 3 = 2)
df_lrs = len(results_model2.data.betaValues) - len(results_model1.data.betaValues)

# Critical value at 5% significance level and p-value
crit_chi2 = chi2.ppf(0.95, df=df_lrs)
p_value = 1 - chi2.cdf(LRS, df=df_lrs)

print(f'LL Model 1 (Restricted):    {LL1:.2f}')
print(f'LL Model 2 (Unrestricted):  {LL2:.2f}')
print(f'LRS:                       {LRS:.2f}')
print(f'Degrees of freedom:        {df_lrs}')
print(f'Critical Chi-square (5%):  {crit_chi2:.2f}')
print(f'p-value:                   {p_value:.4e}')

LL Model 1 (Restricted):    -6737.02
LL Model 2 (Unrestricted):  -6077.74
LRS:                       1318.57
Degrees of freedom:        2
Critical Chi-square (5%):  5.99
p-value:                   0.0000e+00


#### `Answers`

The likelihood-ratio statistic is calculated as $\text{LRS} = -2 \cdot (-6,737.02 - (-6,077.74)) = 1,318.56$. With 2 degrees of freedom, this test statistic exceeds the critical Chi-square value of 5.99 at the 5% significance level. We thus reject the null hypothesis and conclude that Model 2 provides a statistically superior fit.

`10` Based on the test and the ASC estimates, should the ASCs be retained? Explain briefly.

#### `Answers`

The alternative-specific constants should be retained because the alternatives are labelled (train, Swissmetro, and car). In labelled choice contexts, travellers hold intrinsic modal preferences driven by unobserved factors like comfort. Omitting ASCs forces generic parameters to absorb these modal differences, causing bias and a significantly worse model fit. 

### `Part 4: Availability conditions`

Estimate **Model 3**, using the same utility specification as **Model 2**. 

However, now account for the fact that not all alternatives are available in every choice observation, rather than assuming full availability. To do so, use the observed availability conditions (`TRAIN_AV`, `SM_AV`, `CAR_AV`).

In [9]:
# Give a name to the model
model_name = 'Model 3: RUM-MNL with availability conditions'

# Define availability variables
TRAIN_AV = Variable('TRAIN_AV')
SM_AV    = Variable('SM_AV')
CAR_AV   = Variable('CAR_AV')

# Associate availability conditions with the numbering of alternatives
AV = {1: TRAIN_AV, 2: SM_AV, 3: CAR_AV}

# Estimate Model 3 using the utility specification from Model 2
results_model3 = estimate_mnl(V, AV, CHOICE, biodata, model_name)

# Print estimation results
print_results(results_model3)

# Report final log-likelihood
LL_model3 = results_model3.data.logLike
print(f'\nFinal log-likelihood of Model 3: {LL_model3:.2f}')


Estimation finished
Cause of termination           : Relative gradient = 0.00011 <= 0.00012
Number of iterations           :            4
Estimation time                :      0:00:00
Number of estimated parameters :            5
Sample size                    :         6759
Excluded observations          :            0
Null log likelihood            :     -6958.42
Init log likelihood            :     -5624.53
Final log likelihood           :     -5304.02
Rho-square for the null model  :        0.238
Bayesian Information Criterion :     10652.13
Number of threads              :           11

            Value Rob. Std err Rob. t-test Rob. p-value
B_cost    -1.0847       0.0682      -15.91         0.00
B_headway -0.0053       0.0010       -5.34         0.00
B_time    -1.2680       0.1043      -12.16         0.00
asc_sm     0.2662       0.0615        4.33         0.00
asc_train -0.2040       0.0801       -2.55         0.01

Final log-likelihood of Model 3: -5304.02


`11` Compare the final log-likelihood and predicted market shares of Models 1 and 3. Explain how incorrectly assuming full availability affects the model outcomes.

In [10]:
# Function to calculate sample-average predicted market shares
def calc_pred_shares(betas, av_dict):
    b_t = betas['B_time']
    b_c = betas['B_cost']
    b_h = betas['B_headway']
    asc_tr = betas.get('ASC_train', 0)
    asc_sm = betas.get('ASC_sm', 0)
    asc_car = 0
    
    v1 = asc_tr + b_t * df['TRAIN_TT_SCALED'] + b_c * df['TRAIN_COST_SCALED'] + b_h * df['TRAIN_HE']
    v2 = asc_sm + b_t * df['SM_TT_SCALED']    + b_c * df['SM_COST_SCALED']    + b_h * df['SM_HE']
    v3 = asc_car + b_t * df['CAR_TT_SCALED']  + b_c * df['CAR_CO_SCALED']
    
    av1 = av_dict[1] if isinstance(av_dict[1], int) else df['TRAIN_AV']
    av2 = av_dict[2] if isinstance(av_dict[2], int) else df['SM_AV']
    av3 = av_dict[3] if isinstance(av_dict[3], int) else df['CAR_AV']
    
    ev1 = np.exp(v1) * av1
    ev2 = np.exp(v2) * av2
    ev3 = np.exp(v3) * av3
    denom = ev1 + ev2 + ev3
    
    return [
        round((ev1 / denom).mean() * 100, 2),
        round((ev2 / denom).mean() * 100, 2),
        round((ev3 / denom).mean() * 100, 2)
    ]

# Observed sample market shares
obs_shares = [13.30, 60.51, 26.19]


# Predicted shares for each model
pred_m1 = calc_pred_shares(results_model1.get_beta_values(), {1:1, 2:1, 3:1})
pred_m2 = calc_pred_shares(results_model2.get_beta_values(), {1:1, 2:1, 3:1})
pred_m3 = calc_pred_shares(results_model3.get_beta_values(), AV)

# Comparison summary table
df_comparison = pd.DataFrame({
    'Model': [
        'Observed data',
        'Model 1 (Assumed full av, No ASC)',
        'Model 2 (Assumed full av, With ASC)',
        'Model 3 (True AV, with ASC)',
    ],
    'Final LL': ['-', f'{LL_model1:.2f}', f'{LL_model2:.2f}', f'{LL_model3:.2f}'],
    'Null LL':  ['-', f'{results_model1.data.nullLogLike:.2f}', f'{results_model2.data.nullLogLike:.2f}', f'{results_model3.data.nullLogLike:.2f}'],
    'Train [%]': [obs_shares[0], pred_m1[0], pred_m2[0], pred_m3[0]],
    'Swissmetro [%]': [obs_shares[1], pred_m1[1], pred_m2[1], pred_m3[1]],
    'Car [%]': [obs_shares[2], pred_m1[2], pred_m2[2], pred_m3[2]]
})
df_comparison

,Model,Final LL,Null LL,Train [%],Swissmetro [%],Car [%]
0,Observed data,-,-,13.30,60.51,26.19
1,"Model 1 (Assumed full av, No ASC)",-6737.02,-7425.52,18.51,43.53,37.96
2,"Model 2 (Assumed full av, With ASC)",-6077.74,-7425.52,25.75,33.05,41.20
3,"Model 3 (True AV, with ASC)",-5304.02,-6958.42,18.31,53.03,28.66


#### `Answers`

Model 3 achieves a log-likelihood of -5,304.02, representing a  gain of 1,433 points over Model 1 while replicating observed market shares. More importantly, assuming full availability forces the model to assign a non-zero choice probability to the car across 1,161 observations where car option is impossible.

`12` Which model would you retain for behavioural interpretation and forecasting? Justify your answer.

#### `Answers`


Model 3 should be retained for behavioural interpretation and forecasting. Statistically, Model 3 achieves the superior model fit with the highest log-likelihood (-5,304.02) among all evaluated specifications. Behaviourally, it includes alternative-specific constants for a labelled experiment whilst respecting travellers' observed choice sets through availability conditions.

### `Part 5: Socio-demographic interactions`

In this part, we investigate whether sensitivity to travel cost varies across different age groups (taste heterogeneity). 

`13` Recategorise the 5 original age classes into 3 groups:
- **Young** (age ≤ 39) -> `AGE` = 1
- **Middle age** (39 < age ≤ 65) -> `AGE` = 2
- **Old** (age > 65) -> `AGE` = 3



In [11]:
# Recategorise AGE into three groups
df['AGE'] = df['AGE'].map({ 1: 1, 2: 1,
                            3: 2, 4: 2,
                            5: 3})

# Check the distribution
print("Recategorised AGE distribution:")
print(df['AGE'].value_counts().sort_index())

Recategorised AGE distribution:
AGE
1    2367
2    3960
3     432
Name: count, dtype: int64


`14` Estimate **Model 4**, which extends Model 3 by interacting travel cost with age, whilst retaining the alternative-specific constants and observed availability conditions. This model has separate cost parameters (yound, middle age, and old) for each category.


In [12]:
# 2. Reload the Biogeme database with the updated dataframe
biodata = db.Database('swissmetro', df)

# 3. Define the Biogeme Variable for AGE
AGE = Variable('AGE')

# Give a name to the model
model_name = 'Model 4: RUM-MNL with age-cost interaction terms'

# Cost parameters for each recategorised age group
B_cost_young  = Beta('B_cost_young',  0, None, None, 0)   # AGE == 1 (<= 39)
B_cost_middle = Beta('B_cost_middle', 0, None, None, 0)   # AGE == 2 (40 - 65)
B_cost_old    = Beta('B_cost_old',    0, None, None, 0)   # AGE == 5 (> 65)

# Effective travel cost sensitivity interacting with age groups
cost_eff = (B_cost_young  * (AGE == 1) + B_cost_middle * (AGE == 2) + B_cost_old    * (AGE == 3))

# Utility functions
v_train = asc_train + B_time * TRAIN_TT_SCALED + cost_eff * TRAIN_COST_SCALED + B_headway * TRAIN_HE
v_sm    = asc_sm    + B_time * SM_TT_SCALED    + cost_eff * SM_COST_SCALED    + B_headway * SM_HE
v_car   = asc_car   + B_time * CAR_TT_SCALED   + cost_eff * CAR_CO_SCALED

# Associate utility functions and availability conditions
V  = {1: v_train, 2: v_sm, 3: v_car}
AV = {1: TRAIN_AV, 2: SM_AV, 3: CAR_AV}

# Estimate Model 4
results_model4 = estimate_mnl(V, AV, CHOICE, biodata, model_name)

# Print estimation statistics
print_results(results_model4)

# Report final log-likelihood
LL_model4 = results_model4.data.logLike
print(f'\nFinal log-likelihood of Model 4: {LL_model4:.2f}')


Estimation finished
Cause of termination           : Relative gradient = 2.1e-05 <= 0.00012
Number of iterations           :            4
Estimation time                :      0:00:00
Number of estimated parameters :            7
Sample size                    :         6759
Excluded observations          :            0
Null log likelihood            :     -6958.42
Init log likelihood            :     -5610.16
Final log likelihood           :     -5298.13
Rho-square for the null model  :        0.239
Bayesian Information Criterion :     10658.00
Number of threads              :           11

                Value Rob. Std err Rob. t-test Rob. p-value
B_cost_middle -1.1217       0.0684      -16.41         0.00
B_cost_old    -2.0380       0.6102       -3.34         0.00
B_cost_young  -0.9328       0.1339       -6.97         0.00
B_headway     -0.0053       0.0010       -5.36         0.00
B_time        -1.2705       0.1047      -12.14         0.00
asc_sm         0.2642       0.0617      

In [13]:
from scipy.stats import chi2

# Model 3 has 1 generic B_cost, Model 4 has 3 age-specific cost parameters -> df = 2
df_diff = 2
lr_stat = -2 * (LL_model3 - LL_model4)
p_val = 1 - chi2.cdf(lr_stat, df=df_diff)
crit_chi2 = chi2.ppf(0.95, df=df_lrs)
print(f'Likelihood Ratio statistic: {lr_stat:.3f}')
print(f'Degrees of freedom: {df_diff}')
print(f'Critical Chi-square (5%):  {crit_chi2:.2f}')
print(f'p-value: {p_val:.4e}')

Likelihood Ratio statistic: 11.771
Degrees of freedom: 2
Critical Chi-square (5%):  5.99
p-value: 2.7792e-03
